In [1]:
# Import todas las librerias necesarias para EDA
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pprint
from scipy import stats


In [2]:
df_barrios= pd.read_parquet("C:/Users/jrort/OneDrive/Documents/Proyecto segundo semestre/data_bcn/neighborhood_final.parquet")

In [3]:
df_distritos= pd.read_parquet("C:/Users/jrort/OneDrive/Documents/Proyecto segundo semestre/data_bcn/districts_final.parquet")

In [4]:
# ver solo las 
filtro_null_distrito= df_distritos[['num_contracts', 'avg_rent', 'avg_rent_m2', 'avg_surface']].isnull()
print('Filas con valores nulos en variables orginales de distritos:')
print(df_distritos.loc[filtro_null_distrito.any(axis=1),["district", "year", "quarter"]])




Filas con valores nulos en variables orginales de distritos:
                 district  year  quarter
103          Ciutat Vella  2025        4
207              Eixample  2025        4
311        Sants-Montjuïc  2025        4
415             Les Corts  2025        4
519   Sarrià-Sant Gervasi  2025        4
623                Gràcia  2025        4
727        Horta-Guinardó  2025        4
831            Nou Barris  2025        4
935           Sant Andreu  2025        4
1039           Sant Martí  2025        4


In [5]:
# distritos sin valores nulos en variables originales
df_distritos_clean= df_distritos[~filtro_null_distrito.any(axis=1)].copy()

In [6]:
# df_barrios_clean sin año 2025 y quarter 4
df_barrios_clean= df_barrios[~((df_barrios['year'] == 2025) & (df_barrios['quarter'] == 4))].copy()
print(df_barrios_clean.info())
print('Valores nulos en barrios limpios:')
print(df_barrios_clean.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
Index: 3431 entries, 0 to 3502
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   neighborhood_code          3431 non-null   int64         
 1   neighborhood               3431 non-null   object        
 2   year                       3431 non-null   int64         
 3   quarter                    3431 non-null   int64         
 4   num_contracts              3429 non-null   Int64         
 5   avg_rent                   3283 non-null   float64       
 6   avg_rent_m2                3285 non-null   float64       
 7   avg_surface                3400 non-null   float64       
 8   date                       3431 non-null   datetime64[ns]
 9   contract_growth_yoy        3137 non-null   Float64       
 10  rent_growth_yoy            2916 non-null   float64       
 11  rent_m2_growth_yoy         2913 non-null   float64       
 12  contract_gr

In [7]:
# Barrios a excluir del análisis principal a nivel barrio
barrios_problematicos = [
    "Baró de Viver",
    "Can Peguera",
    "Canyelles",
    "Torre Baró",
    "Vallbona",
    "la Clota",
    "la Marina del Prat Vermell - AEI Zona Franca"
]

# Dataset final sin los barrios problemáticos
df_barrios_clean_final = (
    df_barrios_clean.loc[
        ~df_barrios_clean["neighborhood"].isin(barrios_problematicos)
    ]
    .copy()
)



In [8]:
# Copia de trabajo
df_distritos_eda = df_distritos_clean.copy()

# 1) district y quarter como category
df_distritos_eda["district"] = df_distritos_eda["district"].astype("category")
df_distritos_eda["quarter"] = df_distritos_eda["quarter"].astype("category")

# 2) dummies como bool
df_distritos_eda["post_regulation"] = df_distritos_eda["post_regulation"].astype(bool)
df_distritos_eda["covid_dummy"] = df_distritos_eda["covid_dummy"].astype(bool)



In [9]:

# Añadir labels temporales analíticos a df_distritos_eda


# Asegurar quarter como entero para construir periodos
df_distritos_eda["quarter_num"] = df_distritos_eda["quarter"].astype(int)


# 1) Label simple: pre vs post regulación
# Tu dummy post_regulation empieza en 2020Q4

reg_start = pd.Timestamp("2020-10-01")   # 2020Q4

df_distritos_eda["period_prepost"] = np.where(
    df_distritos_eda["date"] < reg_start,
    "Pre-regulation",
    "Post-regulation"
)

df_distritos_eda["period_prepost"] = pd.Categorical(
    df_distritos_eda["period_prepost"],
    categories=["Pre-regulation", "Post-regulation"],
    ordered=True
)


# 2) Label refinado: 4 niveles
# Propuesta corregida:
# - Pre-COVID / pre-regulation: 2000Q1–2019Q4
# - COVID shock: 2020Q1–2020Q3
# - Regulation + COVID / transition: 2020Q4–2021Q4
# - Post-overlap / new regime: 2022Q1–2025Q3

conditions = [
    (df_distritos_eda["year"] <= 2019),

    (df_distritos_eda["year"] == 2020) & (df_distritos_eda["quarter_num"].isin([1, 2, 3])),

    (
        ((df_distritos_eda["year"] == 2020) & (df_distritos_eda["quarter_num"] == 4))
        | (df_distritos_eda["year"] == 2021)
    ),

    (df_distritos_eda["year"] >= 2022)
]

choices = [
    "Pre-COVID / pre-regulation",
    "COVID shock",
    "Regulation + COVID / transition",
    "Post-overlap / new regime"
]

df_distritos_eda["period_4levels"] = np.select(
    conditions,
    choices,
    default="Unknown"
)

df_distritos_eda["period_4levels"] = pd.Categorical(
    df_distritos_eda["period_4levels"],
    categories=[
        "Pre-COVID / pre-regulation",
        "COVID shock",
        "Regulation + COVID / transition",
        "Post-overlap / new regime"
    ],
    ordered=True
)



In [10]:
# Opcional: si no quieres conservar quarter_num
df_distritos_eda.drop(columns="quarter_num", inplace=True)

In [11]:
# Crear dataset de barrios para transformación
df_barrios_eda = df_barrios_clean_final.copy()

# Tipos categóricos (solo si la columna existe)
if "district" in df_barrios_eda.columns:
    df_barrios_eda["district"] = df_barrios_eda["district"].astype("category")
if "neighborhood" in df_barrios_eda.columns:
    df_barrios_eda["neighborhood"] = df_barrios_eda["neighborhood"].astype("category")
if "quarter" in df_barrios_eda.columns:
    df_barrios_eda["quarter"] = df_barrios_eda["quarter"].astype("category")

# Dummies a boolean (solo si existen)
if "post_regulation" in df_barrios_eda.columns:
    df_barrios_eda["post_regulation"] = df_barrios_eda["post_regulation"].astype(bool)
if "covid_dummy" in df_barrios_eda.columns:
    df_barrios_eda["covid_dummy"] = df_barrios_eda["covid_dummy"].astype(bool)

# Labels temporales
df_barrios_eda["quarter_num"] = df_barrios_eda["quarter"].astype(int)

reg_start = pd.Timestamp("2020-10-01")
df_barrios_eda["period_prepost"] = np.where(
    df_barrios_eda["date"] < reg_start,
    "Pre-regulation",
    "Post-regulation"
 )
df_barrios_eda["period_prepost"] = pd.Categorical(
    df_barrios_eda["period_prepost"],
    categories=["Pre-regulation", "Post-regulation"],
    ordered=True
)

conditions = [
    (df_barrios_eda["year"] <= 2019),
    (df_barrios_eda["year"] == 2020) & (df_barrios_eda["quarter_num"].isin([1, 2, 3])),
    (((df_barrios_eda["year"] == 2020) & (df_barrios_eda["quarter_num"] == 4)) | (df_barrios_eda["year"] == 2021)),
    (df_barrios_eda["year"] >= 2022)
 ]

choices = [
    "Pre-COVID / pre-regulation",
    "COVID shock",
    "Regulation + COVID / transition",
    "Post-overlap / new regime"
 ]

df_barrios_eda["period_4levels"] = np.select(
    conditions,
    choices,
    default="Unknown"
 )

df_barrios_eda["period_4levels"] = pd.Categorical(
    df_barrios_eda["period_4levels"],
    categories=[
        "Pre-COVID / pre-regulation",
        "COVID shock",
        "Regulation + COVID / transition",
        "Post-overlap / new regime"
    ],
    ordered=True
)

df_barrios_eda.drop(columns="quarter_num", inplace=True)

print("Columnas disponibles en df_barrios_eda:", df_barrios_eda.columns.tolist())

Columnas disponibles en df_barrios_eda: ['neighborhood_code', 'neighborhood', 'year', 'quarter', 'num_contracts', 'avg_rent', 'avg_rent_m2', 'avg_surface', 'date', 'contract_growth_yoy', 'rent_growth_yoy', 'rent_m2_growth_yoy', 'contract_growth_qoq', 'rent_m2_lag1', 'rent_m2_lag4', 'post_regulation', 'covid_dummy', 'ipc_index_q', 'avg_rent_real_2025base', 'avg_rent_m2_real_2025base', 'rent_real_growth_yoy', 'rent_m2_real_growth_yoy', 'euribor_12m_q', 'ist_index', 'population_total', 'population_density_ha', 'period_prepost', 'period_4levels']


In [12]:
# Función auxiliar para media ponderada
def weighted_mean(values, weights):
    mask = values.notna() & weights.notna()
    if not mask.any():
        return np.nan
    return np.average(values[mask], weights=weights[mask])

# Construir dataset pre vs post por barrio
def build_prepost_unit_dataset(
    df,
    unit_col,
    period_col,
    compare_periods,
    vars_to_compare,
    weight_col="num_contracts"
    ):
    period_pre, period_post = compare_periods

    vars_to_compare = [v for v in vars_to_compare if v in df.columns]

    rows = []

    for (unit, period), g in df.groupby([unit_col, period_col], observed=False):
        if period not in compare_periods:
            continue

        row = {
            unit_col: unit,
            period_col: period
        }

        for var in vars_to_compare:
            if var == "num_contracts":
                row[var] = g[var].mean(skipna=True)
            else:
                row[var] = weighted_mean(g[var], g[weight_col])

        rows.append(row)

    period_df = pd.DataFrame(rows)

    wide = period_df.pivot(index=unit_col, columns=period_col, values=vars_to_compare)

    wide.columns = [f"{var}__{period}" for var, period in wide.columns]
    wide = wide.reset_index()

    for var in vars_to_compare:
        col_pre = f"{var}__{period_pre}"
        col_post = f"{var}__{period_post}"

        if col_pre in wide.columns and col_post in wide.columns:
            wide[f"{var}_diff"] = wide[col_post] - wide[col_pre]

            wide[f"{var}_pct_change"] = np.where(
                wide[col_pre].notna() & wide[col_post].notna() & (wide[col_pre] != 0),
                (wide[col_post] - wide[col_pre]) / wide[col_pre],
                np.nan
            )

    return wide

In [13]:
compare_periods_barrios = (
    "Pre-COVID / pre-regulation",
    "Post-overlap / new regime"
)

vars_hypothesis_barrios = [
    "avg_rent_m2",
    "avg_rent_m2_real_2025base",
    "rent_m2_growth_yoy",
    "rent_m2_real_growth_yoy",
    "num_contracts",
    "contract_growth_yoy"
]




In [16]:
print("Variables disponibles en df_barrios_eda:", df_barrios_eda.info())
print("Variables disponibles en df_distritos_eda:", df_distritos_eda.info())

<class 'pandas.core.frame.DataFrame'>
Index: 3102 entries, 0 to 3502
Data columns (total 28 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   neighborhood_code          3102 non-null   int64         
 1   neighborhood               3102 non-null   category      
 2   year                       3102 non-null   int64         
 3   quarter                    3102 non-null   category      
 4   num_contracts              3102 non-null   Int64         
 5   avg_rent                   3102 non-null   float64       
 6   avg_rent_m2                3102 non-null   float64       
 7   avg_surface                3102 non-null   float64       
 8   date                       3102 non-null   datetime64[ns]
 9   contract_growth_yoy        2838 non-null   Float64       
 10  rent_growth_yoy            2838 non-null   float64       
 11  rent_m2_growth_yoy         2838 non-null   float64       
 12  contract_gr